# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant-python/) library. The dataset describes human protein measurements by mass spectrometry after isolation of extracellular vesicles from stimulated human mast cells.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed in the environment
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and initialize the `mlcroissant.Dataset` object.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import pprint

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
# Create croissant dataset object
dataset = mlc.Dataset(croissant_url)
# Print metadata summary
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

# List dataset version, license, keywords
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview

Inspect available record sets and the fields for this dataset. All entity references use their `@id` values for clarity and reproducibility.

In [ ]:
# Inspect record sets
print('Listing record sets in the dataset:')
record_sets = list(dataset.record_sets())
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '')}")
    print(f"  Description: {rs.get('description', '')}")
    # Print field @ids
    print(f"  Fields:")
    for field in rs.get('fields', []):
        print(f"    - {field['@id']} (name={field.get('name','')}, type={field.get('dataType','')})")
    print()

## 3. Data Extraction

Load data from a record set of interest into a DataFrame using the record set and field `@id` values. For this dataset, we will locate the main record set containing protein abundance and metadata information.

In [ ]:
# Find the canonical record set containing protein tables
main_record_set_id = None
for rs in record_sets:
    # Use heuristics: try description or fields
    fields = [field.get('name','') for field in rs.get('fields',[])]
    field_ids = [field['@id'] for field in rs.get('fields',[])]
    if 'accession' in fields and 'abundance' in fields:
        main_record_set_id = rs['@id']
        break
if main_record_set_id is None:
    # fallback: use first record set
    main_record_set_id = record_sets[0]['@id']
print(f"Selected RecordSet for analysis: {main_record_set_id}")

# List all record_set @ids
all_record_set_ids = [rs['@id'] for rs in record_sets]
print('All RecordSet @ids:', all_record_set_ids)

# Extract data from all record sets
dataframes = {}
for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from RecordSet: {record_set_id}")

# Show columns of the main RecordSet
print("\nFields/Columns in main DataFrame:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's filter and normalize numeric fields, such as protein abundance, remove outliers, and group by protein description if present. 
Reference all columns and fields by their `@id`, as listed previously.

In [ ]:
# Choose a numeric field for analysis - try 'abundance' or similar
df = dataframes[main_record_set_id]
numeric_candidates = [c for c in df.columns if 'abundance' in c or 'count' in c or c.lower().startswith('mw') or c.lower().startswith('coverage')]
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Selected numeric field for EDA: {numeric_field_id}")
else:
    # fallback: use first numeric column
    numeric_cols = df.select_dtypes(include=['number']).columns
    if len(numeric_cols)>0:
        numeric_field_id = numeric_cols[0]
    else:
        raise Exception("No numeric column found for EDA.")

# Remove missing, cast to float
series = pd.to_numeric(df[numeric_field_id], errors='coerce')
df[numeric_field_id+'_float'] = series
# Simple outlier filter: values > mean+3*std
m, s = series.mean(), series.std()
outlier_thresh = m + 3*s
filtered_df = df[series < outlier_thresh]
threshold = m
filtered_high = filtered_df[filtered_df[numeric_field_id+'_float'] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (outlier threshold removed): {len(filtered_high)} rows.")

# Normalize field
filtered_high[numeric_field_id+'_norm'] = (filtered_high[numeric_field_id+'_float'] - filtered_high[numeric_field_id+'_float'].mean())/filtered_high[numeric_field_id+'_float'].std()
print(filtered_high[[numeric_field_id, numeric_field_id+'_float', numeric_field_id+'_norm']].head())

# Group by protein description field if present
group_field_candidates = [c for c in df.columns if 'description' in c or 'protein' in c]
if group_field_candidates:
    group_field_id = group_field_candidates[0]
    grouped_df = filtered_high.groupby(group_field_id)[numeric_field_id+'_float'].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization

Plot the distribution of the selected numeric field and the top proteins by mean abundance.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of abundance (filtered)
plt.figure(figsize=(8,5))
sns.histplot(filtered_high[numeric_field_id+'_float'].dropna(), bins=40, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Barplot of top proteins by normalized abundance, if grouped_df exists
if 'grouped_df' in locals() and grouped_df.shape[0]:
    top_n = grouped_df.sort_values(numeric_field_id+'_float', ascending=False).head(12)
    plt.figure(figsize=(10,5))
    sns.barplot(data=top_n, x=numeric_field_id+'_float', y=group_field_id, palette='viridis')
    plt.title(f"Top proteins by mean {numeric_field_id}")
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(group_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we explored the FAIR2 dataset using the `mlcroissant` library by discovering the record sets, loading the main protein abundance table, and performing basic EDA and visualization. 

- We demonstrated how to reference data elements by their `@id` throughout the process.
- Exploratory analysis identified key numeric fields (e.g., protein abundance), allowed filtering proteins above average abundance, and grouped data by annotation fields (e.g., description).
- A histogram visualized the distribution of protein abundance values, and bar plots highlighted the most abundant proteins.

This approach serves as a template for analyzing any Croissant-conformant dataset using Python and `mlcroissant`.

For further analysis, you may apply machine learning models, additional feature engineering, or integrate other omics data as needed.